# Transfomer

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import itertools
import os
import math
import traceback
import random
import time
import matplotlib.pyplot as plt
from datetime import datetime



In [ ]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)

### DATA LOADING

In [ ]:
df_global=pd.read_csv("Global_5min_interpolated_data.csv") #Inputs
df_local=pd.read_csv("local__irradiance_5min_ambient.csv") #Targets

In [ ]:

# plt.figure(figsize=(15, 5))
# # Plot the first subplot
# plt.plot(df_global["shortwave_radiation_instant (W/m²)"], color='blue')
# # Plot the second subplot
# plt.plot(df_local["Solar Radiation (W/m^2)"], color='green')
# plt.title('Global and Local Irradiance')
# plt.ylabel('Irradiance')
# plt.xlabel('Time(15 MIN)')


### Useful Functions

In [ ]:
# Define the set_seed function to control all randomness
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True) #deterministic matrix computation

#Gradient Loss Function if needed
def gradient_loss(pred, target):
    """
    Computes L1 loss between gradients (finite differences) of prediction and target.
    pred, target: shape (B, T, D)
    """
    pred_grad = pred[:, 1:, :] - pred[:, :-1, :]
    target_grad = target[:, 1:, :] - target[:, :-1, :]
    return torch.mean(torch.abs(pred_grad - target_grad))
    
# ----- Dataset Class -----
class SolarDataset(Dataset):
    def __init__(self, data, seq_len, label_len, pred_len):
        self.seq_len = seq_len
        self.label_len = label_len
        self.pred_len = pred_len
        self.data = data.astype(np.float32)
        self.length = len(data) - seq_len - label_len - pred_len + 1  # <-- adjusted

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]  # encoder input
        y = self.data[idx + self.seq_len : idx + self.seq_len + self.label_len + self.pred_len]  # full decoder target
        return torch.tensor(x), torch.tensor(y)

# ----- Positional Encoding -----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

### Transformer Architecture

In [ ]:
# ----- Model Class -----
class TransformerForecast(nn.Module):
    def __init__(self, input_dim, embed_dim, pred_len,nhead, ff_dim,ff_linear_dim, num_layers):
        
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        self.pred_len = pred_len
        
        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=ff_dim, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=ff_dim, dropout=0.1, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        
        self.encoder_norm = nn.LayerNorm(embed_dim)
        self.decoder_norm = nn.LayerNorm(embed_dim)

        # Extra linear layers to map from embed_dim to input_dim
        self.postprocess = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 192),
            nn.ReLU(),
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
    

    def forward(self, src, tgt):
        
        src = self.input_proj(src)
        src = self.pos_enc(src)#(batch_size, 144, embed_dim)= (32,144,64) for example
        
        tgt = self.input_proj(tgt) 
        tgt = self.pos_enc(tgt)#shape is (32,96+288,64)
    
        memory = self.encoder(src) #(32,144,64)
        out = self.decoder(tgt, memory) #Out shape would be (32,96+288,64)
        

        ####### NEW LINEAR LAYERS #######
        B, T, E = out.shape
        out = out.reshape(-1, E)                         # (B*T, embed_dim)
        out = self.postprocess(out)                      # (B*T, input_dim)
        out = out.reshape(B, T, self.input_dim)          # (B, T, input_dim)

        return out



### Data Preparation

In [ ]:
# ----- RAW DATA ----- #User can change this to select different features or target variable
series = np.stack([
    np.array(df_global["shortwave_radiation_instant (W/m²)"]), 
    np.array(df_local["Solar Radiation (W/m^2)"])
], axis=1)

series_raw = series 

### Hyperparamters

In [ ]:
# ----- CONSTANTS -----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# DEVICE = torch.device("cuda")

param_grid = {
    "SEQ_LEN": [48,96,144,288],
    "EMBED_DIM": [32,64,128],
    "NHEAD": [2],
    "FF_DIM": [256,512,1024],
    "NUM_LAYERS": [1],
    "BATCH_SIZE": [32,96,144,288],
    "LR": [0.001,0.0001,0.00001,0.000001],
    "EPOCHS": [50,200,300],
    "Label_Len": [24,48,96]
}

pred_len=288 # or whatever number of timesteps steps you want to predict into the future
ff_linear_dim=1024
alpha_gradient=0.1
loss_method="Huber" #Huber or Gradient_MSE
optimizer_type ="AdamW" #can be Adam 


In [ ]:
all_combinations = list(itertools.product(*param_grid.values()))
param_names = list(param_grid.keys())

param_names = ["SEQ_LEN", "EMBED_DIM", "NHEAD", "FF_DIM", "NUM_LAYERS", "BATCH_SIZE", "LR", "EPOCHS","Label_Len"]
extra_cols = ["Optimizer","Loss_Method", "RMSE","Train_Time", "Seed", "Run_Number", "Predict_Time", "Loss_History", "Predicted_Local"]
final_columns = param_names + extra_cols

#Change this path if you want to save results in a different location or with a different name
CSV_PATH = "Results_Table/3.2_Decoder-Encoder_Youtube/Grid_Search_Decoder_Encoder_RAW(Fennec)_Version3.csv"

if os.path.exists(CSV_PATH):
    results_df = pd.read_csv(CSV_PATH)
    print(f"Existing file found: Appending to {CSV_PATH}")
    run_number = results_df["Run_Number"].max()
else:
    print(f"Creating new results file: {CSV_PATH}")
    results_df = pd.DataFrame(columns=final_columns)
    run_number = 0


### Training Loop

In [ ]:
# ----- GRID SEARCH LOOP -----
for i, combo in enumerate(all_combinations):
    param_dict = dict(zip(param_names, combo))
    run_number += 1
    
    print(f"\n--- Running config {run_number}/{len(all_combinations)} ---\n{param_dict}")
    print(f"\n--- Running config {run_number}/{len(all_combinations)} ---")

    try:
        # Random seed for reproducibility
        seed = random.randint(0, 2**32 - 1)
        set_seed(seed)

        # Data split
        SEQ_LEN = param_dict["SEQ_LEN"]
        PRED_LEN = pred_len
        LABEL_LEN = param_dict["Label_Len"]
        
        train_data = series[:-PRED_LEN]
        test_raw = series_raw[-(SEQ_LEN + LABEL_LEN + PRED_LEN):]
        test_scaled = series[-(SEQ_LEN + LABEL_LEN + PRED_LEN):]
        

        # Define worker_init_fn for reproducibility
        def worker_init_fn(worker_id):
            np.random.seed(seed + worker_id)  # Ensure each worker gets a different seed
        g = torch.Generator()
        g.manual_seed(seed)
        
        train_dataset = SolarDataset(train_data, SEQ_LEN, LABEL_LEN, PRED_LEN)
        train_loader = DataLoader(train_dataset, batch_size=param_dict["BATCH_SIZE"], shuffle=True,
                                   generator=g, worker_init_fn=worker_init_fn)

        # Model init
        model = TransformerForecast(
            input_dim=2,
            embed_dim=param_dict["EMBED_DIM"],
            nhead=param_dict["NHEAD"],
            ff_dim=param_dict["FF_DIM"],
            ff_linear_dim=ff_linear_dim,
            pred_len = pred_len,
            num_layers=param_dict["NUM_LAYERS"]
        ).to(DEVICE)
        
        if optimizer_type == "Adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=param_dict["LR"])
        elif optimizer_type == "AdamW": 
            optimizer = torch.optim.AdamW(model.parameters(), lr=param_dict["LR"])
        else:
            raise ValueError(f"Unknown Optimizer Type: {optimizer_type}")
            
        if loss_method == "MSE":
            criterion = nn.MSELoss()
        elif loss_method == "Huber":
            criterion = torch.nn.HuberLoss()
        elif loss_method == "Gradient_MSE":
            criterion = nn.MSELoss()  # Used as base; gradient loss is added manually
        else:
            raise ValueError(f"Unknown loss method: {loss_method}")

        
#         scheduler = ExponentialLR(optimizer, gamma=0.9)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                mode='min',             # Minimize the loss
                factor=0.5,             # Reduce LR by half
                patience=2)             # Wait 2 epochs with no improvement
    
        # Training
        loss_history = []
        start_train = time.time()
        for epoch in range(param_dict["EPOCHS"]):
            model.train()
            total_loss = 0
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                #Last known value(Forward Fill)
                decoder_input = torch.zeros_like(y).to(DEVICE)
                decoder_input[:, :LABEL_LEN, :] = y[:, :LABEL_LEN, :] 
                last_value = decoder_input[:, LABEL_LEN - 1:LABEL_LEN, :]  
                decoder_input[:, LABEL_LEN:, :] = last_value.repeat(1, PRED_LEN, 1)
                
                target = y[:, LABEL_LEN:, :] 
                optimizer.zero_grad()
                output = model(x, decoder_input)
                output = output[:, -PRED_LEN:, :]  
                
                ########################## LOSS METHODS ##########################
                
                if loss_method == "Gradient_MSE":
                    loss_mse = criterion(output, target)
                    loss_grad = gradient_loss(output, target)
                    loss = loss_mse + alpha_gradient * loss_grad
                else:
                    loss = criterion(output, target)
                
                ###################################################################
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            scheduler.step(total_loss)
            loss_history.append(total_loss / len(train_loader))
        end_train = time.time()

        # Prediction
        model.eval()
        context = torch.tensor(test_scaled[:SEQ_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        known_decoder = torch.tensor(test_scaled[SEQ_LEN:SEQ_LEN+LABEL_LEN], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        last_value = known_decoder[:, -1:, :]  
        repeated = last_value.repeat(1, PRED_LEN, 1)  
        decoder_input = torch.cat([known_decoder, repeated], dim=1)
        
        
        
        start_pred = time.time()
        with torch.no_grad():
            prediction_scaled = model(context, decoder_input).cpu().squeeze(0).numpy()
        end_pred = time.time()

        prediction_original = prediction_scaled[-PRED_LEN:] 
        true_local = series_raw[-PRED_LEN:, 1]
        predicted_local = prediction_original[:, 1]
        rmse_local = np.sqrt(mean_squared_error(true_local, predicted_local))
        print(f"RMSE {run_number} : {rmse_local}")
        
        # Save results
        row_dict = {
        "SEQ_LEN": param_dict["SEQ_LEN"],
        "EMBED_DIM": param_dict["EMBED_DIM"],
        "NHEAD": param_dict["NHEAD"],
        "FF_DIM": param_dict["FF_DIM"],
        "NUM_LAYERS": param_dict["NUM_LAYERS"],
        "BATCH_SIZE": param_dict["BATCH_SIZE"],
        "LR": param_dict["LR"],
        "EPOCHS": param_dict["EPOCHS"],
        "Seed": seed,
        "Loss_Method": loss_method,
        "Optimizer" : optimizer_type,
        "Run_Number": run_number,
        "Label_Len" : param_dict["Label_Len"],
        "Train_Time": round(end_train - start_train, 4),
        "Predict_Time": round(end_pred - start_pred, 4),
        "RMSE": round(rmse_local, 4),
        "Loss_History": loss_history,
        "Predicted_Local": predicted_local.tolist()
        }


        # Append to DataFrame in correct column order
        row_df = pd.DataFrame([row_dict])[final_columns]
        
        if results_df.empty:
            results_df = row_df
        else:
            results_df = pd.concat([results_df, row_df], ignore_index=True)

        results_df.to_csv(CSV_PATH, index=False)

    except Exception as e:
        print(f"? Error in config {run_number}: {e}")
        traceback.print_exc()
        continue

#### Note: The best model from the results_df would be the one with the lowest RMSE reported on the test dataset